In [ ]:
from google.colab import drive
drive.mount('/content/drive')
!pip -q install python-calamine

import pandas as pd, numpy as np
BASE  = '/content/drive/MyDrive/Palestine_Project'
RAW   = f'{BASE}/00_raw/Palestine_Project_Data.xlsx'
CLEAN = f'{BASE}/01_clean'
ML    = f'{BASE}/02_ml'

def read_sheet(name, **kw):
    return pd.read_excel(RAW, sheet_name=name, engine='calamine', **kw)

def save_clean(df, filename):
    df.to_csv(f'{CLEAN}/{filename}', index=False, encoding='utf-8')
    print(f'saved {filename}: {len(df):,} rows x {df.shape[1]} cols')

In [ ]:
wb = read_sheet('3_WorldBank_Economy')
assert wb.shape == (1498, 70)

theme = {
 'NY.GDP.MKTP.CD':'Output', 'NY.GDP.MKTP.KD.ZG':'Output', 'NY.GDP.PCAP.CD':'Output',
 'FP.CPI.TOTL.ZG':'Prices',
 'SL.UEM.TOTL.NE.ZS':'Labour', 'SL.UEM.TOTL.ZS':'Labour', 'SL.TLF.CACT.ZS':'Labour', 'SL.TLF.CACT.FE.ZS':'Labour',
 'SI.POV.NAHC':'Poverty',
 'SP.POP.TOTL':'Population', 'SP.POP.GROW':'Population', 'SP.URB.TOTL.IN.ZS':'Population',
 'SP.DYN.LE00.IN':'Health', 'SP.DYN.IMRT.IN':'Health', 'SH.XPD.CHEX.GD.ZS':'Health',
 'SE.ADT.LITR.ZS':'Education', 'SE.XPD.TOTL.GD.ZS':'Education',
 'BX.TRF.PWKR.CD.DT':'Trade', 'BN.CAB.XOKA.CD':'Trade', 'NE.TRD.GNFS.ZS':'Trade', 'NE.EXP.GNFS.ZS':'Trade', 'NE.IMP.GNFS.ZS':'Trade',
 'EG.ELC.ACCS.ZS':'Services', 'SH.H2O.BASW.ZS':'Services'}
sel = wb[wb['Indicator Code'].isin(theme)].copy()
assert len(sel) == len(theme) == 24

years = [c for c in wb.columns if str(c).isdigit()]
long = sel.melt(id_vars=['Indicator Code', 'Indicator Name'], value_vars=years, var_name='year', value_name='value')
long['year'] = long['year'].astype(int)
long = long[long['value'].notna()].rename(columns={'Indicator Code': 'indicator_code'})

long['is_unreliable_flag'] = (long['indicator_code'].isin(['EG.ELC.ACCS.ZS', 'SH.H2O.BASW.ZS']) & (long['year'] >= 2023)).astype(int)
long['needs_review_flag']  = ((long['indicator_code'] == 'FP.CPI.TOTL.ZG') & (long['year'] == 2024)).astype(int)
fact = long[['indicator_code', 'year', 'value', 'is_unreliable_flag', 'needs_review_flag']].sort_values(['indicator_code', 'year'])

dim = pd.DataFrame({'indicator_code': sel['Indicator Code'], 'indicator_name': sel['Indicator Name']})
dim['theme'] = dim['indicator_code'].map(theme); dim['source'] = 'WB'

print('raw indicator rows', len(wb), '| chosen', len(dim), '| fact rows', len(fact))
n = fact[(fact['year'].between(2016, 2025))].groupby('indicator_code').size()
print('values in 2016-2025 per indicator (min / max):', int(n.min()), '/', int(n.max()))
print('SP.POP.TOTL 2023 =', int(fact[(fact.indicator_code == 'SP.POP.TOTL') & (fact.year == 2023)]['value'].iloc[0]))
print('GDP 2024 (billion US$) =', round(fact[(fact.indicator_code == 'NY.GDP.MKTP.CD') & (fact.year == 2024)]['value'].iloc[0] / 1e9, 2))
print('inflation 2024 (needs review) =', round(fact[(fact.indicator_code == 'FP.CPI.TOTL.ZG') & (fact.year == 2024)]['value'].iloc[0], 2))
save_clean(dim, 'dim_indicator.csv'); save_clean(fact, 'fact_worldbank.csv')